In [1]:
import pandas as pd
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
con = duckdb.connect()

In [5]:
query_orders = """
SELECT 
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_source,
    c.age_group,
    c.acquisition_channel,
    p.payment_value AS order_value
FROM '../../../data/transaction/orders.csv' o
JOIN '../../../data/master/customers.csv' c
ON o.customer_id = c.customer_id
JOIN '../../../data/transaction/payments.csv' p
ON o.order_id = p.order_id
"""
df_orders = con.execute(query_orders).df()
df_orders

,order_id,customer_id,order_date,order_source,age_group,acquisition_channel,order_value
0,1,58578,2012-07-04,paid_search,35-44,social_media,7967.54
1,2,58621,2012-07-04,paid_search,18-24,social_media,71163.75
2,3,58811,2012-07-04,direct,35-44,direct,33660.99
3,4,59453,2012-07-04,referral,45-54,direct,53196.25
4,6,57821,2012-07-06,email_campaign,18-24,social_media,1597.84
...,...,...,...,...,...,...,...
646940,581183,139428,2018-04-26,paid_search,18-24,social_media,21341.30
646941,581184,132628,2018-04-26,paid_search,25-34,paid_search,35204.25
646942,581186,133479,2018-04-24,social_media,18-24,social_media,5888.08
646943,581187,133541,2018-04-27,email_campaign,18-24,organic_search,25611.96


In [6]:
df_cust = df_orders.groupby("customer_id").agg({
    "order_id": "count",
    "order_value": ["sum", "mean"],
    "order_date": ["min", "max"]
}).reset_index()

df_cust.columns = ["customer_id", "num_orders", "total_value", "avg_order_value", "first_order", "last_order"]
df_cust 

,customer_id,num_orders,total_value,avg_order_value,first_order,last_order
0,1,6,142803.47,23800.578333,2012-07-25,2021-04-24
1,2,4,204693.89,51173.472500,2013-09-20,2022-07-06
2,3,3,52093.47,17364.490000,2012-08-27,2013-07-29
3,4,1,10939.06,10939.060000,2020-06-28,2020-06-28
4,5,5,64179.86,12835.972000,2012-08-09,2019-03-27
...,...,...,...,...,...,...
90241,157554,1,6263.81,6263.810000,2022-05-17,2022-05-17
90242,157555,2,95613.85,47806.925000,2015-03-01,2019-06-19
90243,157557,1,5670.72,5670.720000,2017-06-02,2017-06-02
90244,157561,22,544675.90,24757.995455,2012-07-18,2020-08-24
